[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/rag-pipeline-practice/03_document_structuring/03_document_structuring_solutions.ipynb)

# 03. 문서 구조화 실습 — 연습 문제 해설

[03_document_structuring.ipynb](03_document_structuring.ipynb) 끝의 연습 문제 2개에 대한 정답 코드와 해설입니다. **먼저 직접 시도해본 뒤** 참고하세요.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    !pip install -q pydantic openai python-dotenv

In [ ]:
import os
import re
from pydantic import BaseModel, Field

## 연습 1. `urgency` 필드 추가

In [ ]:
class RegulationInquiryV2(BaseModel):
    document_type: str = Field(description="서류의 종류")
    applicant_request: str = Field(description="신청자가 요청하는 핵심 내용")
    related_dates: list[str] = Field(default_factory=list, description="서류에 등장하는 날짜들")
    keywords: list[str] = Field(default_factory=list, description="핵심 키워드 목록")
    urgency: str = Field(default="보통", description="문서의 긴급도: 높음/보통/낮음")


URGENT_WORDS = ["긴급", "즉시", "당일", "오늘까지"]


def structure_with_rules_v2(raw_text: str) -> RegulationInquiryV2:
    doc_type = "휴가신청서" if "휴가" in raw_text else "미분류 문서"

    reason_match = re.search(r"사\s*유\s*:\s*(.+)", raw_text)
    reason = reason_match.group(1).strip() if reason_match else ""

    keywords = [kw for kw in ["연차휴가", "육아휴직", "재택근무", "경조휴가"] if kw in raw_text]

    urgency = "높음" if any(w in raw_text for w in URGENT_WORDS) else "보통"

    return RegulationInquiryV2(
        document_type=doc_type, applicant_request=reason, keywords=keywords, urgency=urgency
    )


normal_text = "휴가 신청서\n사 유 : 개인 사정으로 인한 여름 휴가 사용"
urgent_text = "휴가 신청서\n사 유 : 긴급 병가로 즉시 승인 요청"

print(structure_with_rules_v2(normal_text).model_dump_json(indent=2))
print(structure_with_rules_v2(urgent_text).model_dump_json(indent=2))

**해설**
- Pydantic 필드에 `default="보통"`을 주면, 규칙이 아무것도 걸리지 않았을 때도 값이 비어있지 않고 안전한 기본값으로 채워집니다.
- 실제 OpenAI 호출로 바꾸더라도 할 일은 `RegulationInquiry` 대신 `RegulationInquiryV2`를 `response_format`에 넘기는 것뿐입니다 — 필드에 붙인 `description`이 모델에게 "긴급도를 어떻게 판단할지"에 대한 힌트가 되므로, 규칙 기반 버전보다 더 정교하게 판단할 수 있습니다.

## 연습 2. `structure_text()`를 배치로 호출하는 함수

실습 3에서 정의한 `structure_text()`(키가 있으면 OpenAI, 없으면 규칙 기반으로 자동 전환)를 그대로 재사용합니다 — 배치 함수가 별도의 구조화 로직을 새로 만들지 않고, 이미 검증된 `structure_text()`에 위임하는 것이 핵심입니다.

In [ ]:
class RegulationInquiry(BaseModel):
    document_type: str = Field(description="서류의 종류")
    applicant_request: str = Field(description="신청자가 요청하는 핵심 내용")
    related_dates: list[str] = Field(default_factory=list)
    keywords: list[str] = Field(default_factory=list)


SYSTEM_PROMPT = (
    "당신은 사내 서류 텍스트를 분석해 정해진 형식의 데이터로 정리하는 도우미입니다. "
    "텍스트에 없는 내용은 추측해서 지어내지 말고, 알 수 없으면 빈 값으로 두세요."
)


def structure_with_openai(raw_text: str) -> RegulationInquiry:
    from openai import OpenAI

    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    completion = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": raw_text},
        ],
        response_format=RegulationInquiry,
    )
    return completion.choices[0].message.parsed


def structure_with_rules(raw_text: str) -> RegulationInquiry:
    doc_type = "휴가신청서" if "휴가" in raw_text else "미분류 문서"
    reason_match = re.search(r"사\s*유\s*:\s*(.+)", raw_text)
    reason = reason_match.group(1).strip() if reason_match else ""
    keywords = [kw for kw in ["연차휴가", "육아휴직", "재택근무", "경조휴가"] if kw in raw_text]
    return RegulationInquiry(document_type=doc_type, applicant_request=reason, keywords=keywords)


def structure_text(raw_text: str) -> RegulationInquiry:
    if os.getenv("OPENAI_API_KEY"):
        return structure_with_openai(raw_text)
    return structure_with_rules(raw_text)


def structure_batch(raw_texts: list[str]) -> list[RegulationInquiry]:
    """structure_text() 하나를 여러 건에 반복 적용 — 키 유무에 따른 자동 전환은 structure_text()에 위임한다."""
    results = []
    for i, text in enumerate(raw_texts):
        try:
            results.append(structure_text(text))
        except Exception as e:
            # 한 건이 실패해도 배치 전체가 멈추지 않도록 한다 (crawl.py의 에러 처리 방식과 같은 패턴)
            print(f"[{i}] 구조화 실패: {e}")
    return results


batch_texts = [
    "휴가 신청서\n사 유 : 개인 사정으로 인한 여름 휴가 사용",
    "재직증명서 발급 신청\n사 유 : 은행 대출 서류 제출용",
    "육아휴직 신청서\n사 유 : 만 2세 자녀 양육을 위한 육아휴직 신청",
]

batch_results = structure_batch(batch_texts)
for r in batch_results:
    print(r.model_dump_json())

**해설**
- 배치 처리에서는 한 건의 텍스트가 이상하다고 전체가 멈추면 안 되므로, 개별 건마다 `try/except`로 감싸서 실패한 건은 건너뛰고 계속 진행하도록 만들었습니다. `crawl.py`가 URL 하나가 실패해도 다음 URL로 넘어가는 것과 같은 설계 원칙입니다.
- 실제 OpenAI 버전으로 바꿀 때는 API 호출 비용/속도 때문에 배치를 한 번에 하나씩 순차 호출하기보다 동시 요청(예: `asyncio` + `AsyncOpenAI`)을 고려하는 경우가 많습니다. 다만 이 노트북 범위에서는 순차 처리로 충분합니다.